# Step 12 — ERA5 MJO Download (u850, u200, OLR)
**Project:** ENSO-BSISO Self-Supervised Learning — MJO Extension  
**Author:** Jiayi (jh9141@nyu.edu)

Downloads ERA5 fields needed for the Wheeler & Hendon (2004) RMM preprocessing pipeline:
- `u850` — zonal wind at 850 hPa  
- `u200` — zonal wind at 200 hPa  
- `OLR`  — top net thermal radiation (`ttr`)

**Key differences from the BSISO download (nb01b):**
| Aspect | BSISO (nb01b) | MJO (this notebook) |
|--------|--------------|--------------------|
| Wind variables | u850 + **v850** | u850 + **u200** |
| Domain | 60°E–160°E, 0°–60°N | **Global, 15°S–15°N** |
| Months | MJJAS | **All 12 months** (all-year) |
| N days | ~6,600 | **~16,425** |

**Domain:** 15°S–15°N, all longitudes (−180° to 180°), 2° resolution  
**Period:** All months, 1979–2023  
**Output files (Google Drive → `BSISO_SSL_Project/MJO/data/raw/`):**
```
u850_u200_1979_1988.nc   ← both pressure levels in one file, 5 year-chunks
u850_u200_1989_1998.nc
u850_u200_1999_2008.nc
u850_u200_2009_2018.nc
u850_u200_2019_2023.nc
OLR_MJO_1979_2023.nc     ← single file, all years
```

**Estimated download time:** 40–80 min total (CDS queue dependent)  
**Estimated file sizes:** ~3–6 MB per wind chunk; ~80–120 MB for OLR

---
⚠️ **Prerequisite:** CDS API key from https://cds.climate.copernicus.eu/ (same account as BSISO project)

## Cell 1 — Mount Google Drive + Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_DIR = '/content/drive/MyDrive/BSISO_SSL_Project'
MJO_RAW_DIR = f'{PROJECT_DIR}/MJO/data/raw'

os.makedirs(MJO_RAW_DIR, exist_ok=True)

print('Google Drive mounted.')
print(f'MJO raw data folder: {MJO_RAW_DIR}')
print('Files currently in MJO/data/raw/:')
for f in sorted(os.listdir(MJO_RAW_DIR)):
    mb = os.path.getsize(f'{MJO_RAW_DIR}/{f}') / 1e6
    print(f'  {f}  ({mb:.1f} MB)')

## Cell 2 — Install CDS API Client

In [ ]:
!pip install cdsapi --quiet
import cdsapi
print('cdsapi ready.')

## Cell 3 — Set Up CDS API Credentials

In [ ]:
# ============================================================
# FILL IN YOUR PERSONAL ACCESS TOKEN HERE
# Find it at: https://cds.climate.copernicus.eu/ → Your profile
# ============================================================
CDS_API_KEY = 'YOUR_CDS_API_KEY_HERE'
# ============================================================

cdsapirc = f'url: https://cds.climate.copernicus.eu/api\nkey: {CDS_API_KEY}\n'
with open(os.path.expanduser('~/.cdsapirc'), 'w') as f:
    f.write(cdsapirc)

print('CDS credentials saved.')

try:
    client = cdsapi.Client(quiet=True)
    print('CDS API connection: OK')
except Exception as e:
    print(f'CDS API connection FAILED: {e}')

## Cell 4 — Download u850 + u200 (5 Year-Chunks)

Both pressure levels (850 hPa and 200 hPa) are downloaded together in a single CDS request per chunk.  
Domain: **global equatorial strip 15°S–15°N**  
Grid: 2° × 2° → ~15 lat × 180 lon per level per day

In [ ]:
import cdsapi
import os

if not os.path.exists(os.path.expanduser('~/.cdsapirc')):
    raise RuntimeError('Run Cell 3 first to set up CDS credentials.')

client = cdsapi.Client()

ALL_MONTHS = [f'{m:02d}' for m in range(1, 13)]
DAYS       = [f'{d:02d}' for d in range(1, 32)]  # CDS ignores invalid dates

# 5-year chunks to avoid CDS timeout
year_chunks = [
    (1979, 1988),
    (1989, 1998),
    (1999, 2008),
    (2009, 2018),
    (2019, 2023),
]

for start_yr, end_yr in year_chunks:
    out_file = f'{MJO_RAW_DIR}/u850_u200_{start_yr}_{end_yr}.nc'

    if os.path.exists(out_file):
        size_mb = os.path.getsize(out_file) / 1e6
        print(f'[SKIP] {os.path.basename(out_file)} already exists ({size_mb:.1f} MB)')
        continue

    print(f'Downloading u850+u200  {start_yr}–{end_yr}  all months ... (do NOT close tab)')

    client.retrieve(
        'reanalysis-era5-pressure-levels',
        {
            'product_type'  : 'reanalysis',
            'variable'      : 'u_component_of_wind',
            'pressure_level': ['850', '200'],          # both levels in one request
            'year'          : [str(y) for y in range(start_yr, end_yr + 1)],
            'month'         : ALL_MONTHS,
            'day'           : DAYS,
            'time'          : '12:00',
            'area'          : [15, -180, -15, 180],    # N, W, S, E — global 15°S-15°N
            'grid'          : [2.0, 2.0],
            'data_format'   : 'netcdf',
        },
        out_file
    )

    size_mb = os.path.getsize(out_file) / 1e6
    print(f'  Saved: {os.path.basename(out_file)}  ({size_mb:.1f} MB)\n')

print('All wind chunks done.')

## Cell 5 — Download OLR (All Years, Single Request)

Same variable (`top_net_thermal_radiation`) as BSISO project but over the global equatorial strip.  
All 45 years in one request (OLR file is smaller than wind since it has only 1 level).

In [ ]:
out_olr = f'{MJO_RAW_DIR}/OLR_MJO_1979_2023.nc'

if os.path.exists(out_olr):
    size_mb = os.path.getsize(out_olr) / 1e6
    print(f'[SKIP] OLR_MJO_1979_2023.nc already exists ({size_mb:.1f} MB)')
else:
    print('Downloading OLR  1979–2023  all months ... (do NOT close tab)')

    client.retrieve(
        'reanalysis-era5-single-levels',
        {
            'product_type': 'reanalysis',
            'variable'    : 'top_net_thermal_radiation',
            'year'        : [str(y) for y in range(1979, 2024)],
            'month'       : ALL_MONTHS,
            'day'         : DAYS,
            'time'        : '12:00',
            'area'        : [15, -180, -15, 180],
            'grid'        : [2.0, 2.0],
            'data_format' : 'netcdf',
        },
        out_olr
    )

    size_mb = os.path.getsize(out_olr) / 1e6
    print(f'Saved: OLR_MJO_1979_2023.nc  ({size_mb:.1f} MB)')

## Cell 6 — Verify Downloads

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd

print('=' * 65)
print('VERIFICATION REPORT — MJO ERA5 Downloads')
print('=' * 65)

# --- Wind chunks ---
print('\n[1] Wind files (u850 + u200)')
wind_files = sorted([
    f'{MJO_RAW_DIR}/{f}' for f in os.listdir(MJO_RAW_DIR)
    if f.startswith('u850_u200') and f.endswith('.nc')
])

ds_wind_list = []
for wf in wind_files:
    ds = xr.open_dataset(wf)
    n_times  = len(ds.valid_time)
    t_min    = str(ds.valid_time.values[0])[:10]
    t_max    = str(ds.valid_time.values[-1])[:10]
    n_levels = len(ds.pressure_level)
    size_mb  = os.path.getsize(wf) / 1e6
    print(f'  {os.path.basename(wf):35s}  {n_times:4d} days  {n_levels} levels  '
          f'{t_min} → {t_max}  ({size_mb:.0f} MB)')
    ds_wind_list.append(ds)

ds_wind_all = xr.concat(ds_wind_list, dim='valid_time')
total_days  = len(ds_wind_all.valid_time)
n_lat       = len(ds_wind_all.latitude)
n_lon       = len(ds_wind_all.longitude)

print(f'\n  Combined: {total_days} time steps  (expected ~16,425 for 45 years all-month)')
print(f'  Grid:     {n_lat} lat × {n_lon} lon  (expected ~15 lat, 180 lon at 2°)')
print(f'  Levels:   {sorted(ds_wind_all.pressure_level.values.tolist())} hPa  (expected [200, 850])')
months_found = sorted(set(pd.DatetimeIndex(ds_wind_all.valid_time.values).month))
print(f'  Months:   {months_found}  (expected all 12)')

u850_vals = ds_wind_all['u'].sel(pressure_level=850).values
u200_vals = ds_wind_all['u'].sel(pressure_level=200).values
print(f'\n  u850 range: [{u850_vals.min():.1f}, {u850_vals.max():.1f}] m/s')
print(f'  u200 range: [{u200_vals.min():.1f}, {u200_vals.max():.1f}] m/s')

# --- OLR ---
print('\n[2] OLR file')
ds_olr  = xr.open_dataset(out_olr)
n_olr   = len(ds_olr.valid_time)
t_min   = str(ds_olr.valid_time.values[0])[:10]
t_max   = str(ds_olr.valid_time.values[-1])[:10]
size_mb = os.path.getsize(out_olr) / 1e6
n_lat_o = len(ds_olr.latitude)
n_lon_o = len(ds_olr.longitude)
print(f'  OLR_MJO_1979_2023.nc  {n_olr} days  {n_lat_o} lat × {n_lon_o} lon  '
      f'{t_min} → {t_max}  ({size_mb:.0f} MB)')
months_olr = sorted(set(pd.DatetimeIndex(ds_olr.valid_time.values).month))
print(f'  Months: {months_olr}  (expected all 12)')

# --- NaN check ---
nan_u850 = int(np.isnan(u850_vals).sum())
nan_u200 = int(np.isnan(u200_vals).sum())
nan_olr  = int(np.isnan(ds_olr['ttr'].values).sum())
print(f'\n[3] NaN counts')
print(f'  u850: {nan_u850}  (expected 0)')
print(f'  u200: {nan_u200}  (expected 0)')
print(f'  OLR:  {nan_olr}   (expected 0)')

# --- Date alignment check ---
wind_dates = set(pd.DatetimeIndex(ds_wind_all.valid_time.values).normalize())
olr_dates  = set(pd.DatetimeIndex(ds_olr.valid_time.values).normalize())
missing_olr  = wind_dates - olr_dates
missing_wind = olr_dates  - wind_dates
print(f'\n[4] Date alignment')
print(f'  Dates in wind but not OLR:  {len(missing_olr)}')
print(f'  Dates in OLR but not wind:  {len(missing_wind)}')
if missing_olr or missing_wind:
    print('  WARNING: date mismatch — check downloads')
else:
    print('  OK: wind and OLR cover identical dates')

print('\nVerification complete.')

## Cell 7 — Quick Plot (Visual Sanity Check)

Plots a boreal winter (January) and boreal summer (July) day to verify the known MJO-band climatology:  
- **u200** should show strong subtropical jet (westerlies) at higher latitudes, weaker at equator  
- **u850** shows low-level easterlies in tropics year-round  
- **OLR** should show equatorial convection concentrated over the warm pool (Indian Ocean/West Pacific)

In [ ]:
import matplotlib.pyplot as plt

times_wind = pd.DatetimeIndex(ds_wind_all.valid_time.values)
times_olr  = pd.DatetimeIndex(ds_olr.valid_time.values)

# Pick one January and one July day (first occurrence)
idx_jan_w = int(np.where(times_wind.month == 1)[0][0])
idx_jul_w = int(np.where(times_wind.month == 7)[0][0])
idx_jan_o = int(np.where(times_olr.month  == 1)[0][0])
idx_jul_o = int(np.where(times_olr.month  == 7)[0][0])

lats = ds_wind_all.latitude.values
lons = ds_wind_all.longitude.values

u850_jan = ds_wind_all['u'].isel(valid_time=idx_jan_w).sel(pressure_level=850).values
u850_jul = ds_wind_all['u'].isel(valid_time=idx_jul_w).sel(pressure_level=850).values
u200_jan = ds_wind_all['u'].isel(valid_time=idx_jan_w).sel(pressure_level=200).values
u200_jul = ds_wind_all['u'].isel(valid_time=idx_jul_w).sel(pressure_level=200).values
olr_jan  = -ds_olr['ttr'].isel(valid_time=idx_jan_o).values  # flip sign: ttr<0 = OLR>0
olr_jul  = -ds_olr['ttr'].isel(valid_time=idx_jul_o).values

date_jan_w = str(times_wind[idx_jan_w])[:10]
date_jul_w = str(times_wind[idx_jul_w])[:10]
date_jan_o = str(times_olr[idx_jan_o])[:10]
date_jul_o = str(times_olr[idx_jul_o])[:10]

fig, axes = plt.subplots(3, 2, figsize=(18, 11))
fig.suptitle('MJO ERA5 Sanity Check — global equatorial strip (15°S–15°N)', fontsize=14, fontweight='bold')

panels = [
    (u850_jan, f'u850  Jan ({date_jan_w})',  'RdBu_r', 'm/s'),
    (u850_jul, f'u850  Jul ({date_jul_w})',  'RdBu_r', 'm/s'),
    (u200_jan, f'u200  Jan ({date_jan_w})',  'RdBu_r', 'm/s'),
    (u200_jul, f'u200  Jul ({date_jul_w})',  'RdBu_r', 'm/s'),
    (olr_jan,  f'OLR   Jan ({date_jan_o})',  'RdBu_r', 'J/m²'),
    (olr_jul,  f'OLR   Jul ({date_jul_o})',  'RdBu_r', 'J/m²'),
]

for ax, (data, title, cmap, unit) in zip(axes.flat, panels):
    im = ax.imshow(data, cmap=cmap, origin='upper',
                   extent=[lons.min(), lons.max(), lats.min(), lats.max()],
                   aspect='auto')
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.axhline(0, color='k', lw=0.5, alpha=0.5)
    plt.colorbar(im, ax=ax, label=unit, shrink=0.8)

plt.tight_layout()
fig_path = f'{PROJECT_DIR}/MJO/results/era5_mjo_sanity_check.png'
plt.savefig(fig_path, dpi=130, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')
print()
print('Expected patterns:')
print('  u850: low-level easterlies (negative) over most of tropics')
print('  u200: upper-level westerlies (positive) over tropics, reversed from u850')
print('  OLR:  strong convection (large negative ttr = positive OLR) over warm pool')

---
## Done!

Google Drive should now contain:

```
BSISO_SSL_Project/MJO/data/raw/
├── rmm_labels.csv              ← from nb11
├── nino34_monthly.txt          ← from nb11
├── rmm_raw.txt                 ← from nb11
├── u850_u200_1979_1988.nc      ← from this notebook ✓
├── u850_u200_1989_1998.nc
├── u850_u200_1999_2008.nc
├── u850_u200_2009_2018.nc
├── u850_u200_2019_2023.nc
└── OLR_MJO_1979_2023.nc
```

**Next step:** Run `13_mjo_preprocessing.ipynb`  
Requires both nb11 (`rmm_labels.csv`) and nb12 (ERA5 files) to be complete.

---
*DDCS Project | jh9141@nyu.edu*